In [1]:
# Select files

from os import listdir, getenv
from os.path import isfile, join
from collections import namedtuple
from dataclasses import dataclass, field
from docling.datamodel.document import DoclingDocument
from pymupdf import Document
from dotenv import load_dotenv

load_dotenv()

@dataclass
class FileInfo:
    fname: str
    title: str | None = None
    ddoc: DoclingDocument | None = None
    pdoc: Document | None = None
    mlvl: int = 0
    ptoc: list = field(default_factory = list)
    tree_root: list = field(default_factory=list)

files_to_ingest = []
docs_path = "../docs"
files = sorted(f for f in listdir(docs_path) 
               if isfile(join(docs_path, f)) and f.lower().endswith(".pdf")
              )
file_count = len(files)

print(f"\nSelecting {file_count} files from {docs_path} folder:\n");

for fidx in range(len(files)):
    print(f"{fidx}: '{files[fidx]}'")
    files[fidx] = f"{docs_path}/{files[fidx]}"
    files_to_ingest.append(FileInfo(files[fidx]))

print("\nSelected files:")
print([item.fname for item in files_to_ingest])# Parse the files


Selecting 15 files from ../docs folder:

0: 'Algorithms-JeffE.pdf'
1: 'An-Open-Guide-to-Data-Structures-and-Algorithms-1743008152._print.pdf'
2: 'Computer Organization and Assembly Language Programming.pdf'
3: 'Foundations_of_Information_Systems_-_WEB_oNlbGYl.pdf'
4: 'OER-202301_Wang_2023-Introduction-to-Computer-Programming-with-Python.pdf'
5: 'Pro Git.pdf'
6: 'Python for Everybody.pdf'
7: 'byte-of-python.pdf'
8: 'git-internals.pdf'
9: 'ods-cpp.pdf'
10: 'ods-java.pdf'
11: 'ods-python.pdf'
12: 'thinkpython1.pdf'
13: 'thinkpython2.pdf'
14: 'x86-64 Assembly Language Programming with Ubuntu.pdf'

Selected files:
['../docs/Algorithms-JeffE.pdf', '../docs/An-Open-Guide-to-Data-Structures-and-Algorithms-1743008152._print.pdf', '../docs/Computer Organization and Assembly Language Programming.pdf', '../docs/Foundations_of_Information_Systems_-_WEB_oNlbGYl.pdf', '../docs/OER-202301_Wang_2023-Introduction-to-Computer-Programming-with-Python.pdf', '../docs/Pro Git.pdf', '../docs/Python for Every

In [ ]:
import onnxruntime as ort
from docling.document_converter import DocumentConverter
import pymupdf
import pprint

converter = DocumentConverter()

# Parse the files
for index, pdf_file in enumerate(files_to_ingest):
    print(f"Converting {pdf_file.fname}...")
    result = converter.convert(pdf_file.fname)

    print(f"Converted {pdf_file.fname}")
    pdf_file.ddoc = result.document

    print(f"Name: '{pdf_file.ddoc.name}'")
    print(f"Pages: {len(pdf_file.ddoc.pages)}")
    print(f"Pictures: {len(pdf_file.ddoc.pictures)}")
    print(f"Tables: {len(pdf_file.ddoc.tables)}")
    print()
    
    pdf_file.pdoc = pymupdf.open(pdf_file.fname)

    toc = pdf_file.pdoc.get_toc()#simple=False)
    pdf_file.ptoc = pdf_file.pdoc.get_toc(simple=False)

    if pdf_file.pdoc.metadata["title"]:
        pdf_file.title = pdf_file.pdoc.metadata["title"]
    elif pdf_file.ddoc.name:
        pdf_file.title = pdf_file.ddoc.name
        
    pprint.pp(f"Document title: {pdf_file.title}")
    print()

    for index, item in enumerate(pdf_file.ptoc):
        print(f"[{index}] {"  " * (item[0] - 1)}{item[1]} {"." * (70 - len(item[1]) - (item[0] - 1) * 2)} {item[3].get('page', 'na')}")
        if item[0] > pdf_file.mlvl:
            pdf_file.mlvl = item[0]
            
    print("\n\n")

In [ ]:
# Clear resources

import gc, torch

print(torch.cuda.memory_allocated() / 1e6, "MB in use by tensors")
print(torch.cuda.memory_reserved() / 1e6, "MB held by the cache")

print("Cleaning resources...")

del converter
gc.collect()

torch.cuda.synchronize()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_allocated() / 1e6, "MB in use by tensors")
print(torch.cuda.memory_reserved() / 1e6, "MB held by the cache")

In [ ]:
from docling_core.types.doc import BoundingBox, CoordOrigin, NodeItem, SectionHeaderItem, TextItem
from difflib import SequenceMatcher
import unicodedata
from docling_core.transforms.serializer.markdown import MarkdownDocSerializer
from typing import TypeAlias
import numpy as np

def get_ditem_md(pdf_file, item):
    serializer = MarkdownDocSerializer(doc=pdf_file.ddoc)
    serialized_output = serializer.serialize(item=item)
    return serialized_output.text


def is_similar_title(a, b):
    if a in b or b in a:
        return True
        
    if SequenceMatcher(None, a, b).ratio() >= 0.75:
        return True

    return False


def get_sections(pdf_file, section_idx):
    section = pdf_file.ptoc[section_idx];
        
    if section_idx == len(pdf_file.ptoc) - 1:
        next_section = None
        end_page = len(pdf_file.ddoc.pages) - 1
    else:
        next_section_idx = section_idx + 1
        next_section = pdf_file.ptoc[next_section_idx]
        end_page = pdf_file.ptoc[next_section_idx][2]-1
    
    start_page=pdf_file.ptoc[section_idx][2]-1
    
    section_text = section[1].lower().strip()
    
    if next_section:
        next_section_text = next_section[1].lower().strip()
    else:
        next_section_text = None    
    
    content_sections = []
    get_out = False
    started = False
    
    for item, _ in pdf_file.ddoc.iterate_items():
        text = getattr(item, "text", "").strip()
        level = getattr(item, "level", None)
        prov = getattr(item, "prov", None) or []
    
        if not prov:
            continue
    
        for prov_item in prov:
            bbox = getattr(prov_item, "bbox", None)
            page_no = prov_item.page_no
    
            if bbox is None:
                continue
                
            if not (start_page < page_no <= (end_page + 1)):
                continue
            
            # print(text)
                            
            if level is not None and level>0:
                # print(f"-" * len(text))
                # print(prov_item)
                
                ltext = text.lower()
    
                # TODO better selection logic using location if possible
                
                if content_sections and next_section_text:
                    if is_similar_title(ltext, next_section_text):
                        # print(">>>> END")
                        get_out = True
                        break
                
                if not content_sections and page_no == start_page + 1:
                    if is_similar_title(ltext, section_text):
                        # print("<<<< START")
                        started = True
                else:
                    # candidate page passed but section not found
                    if not content_sections:
                        get_out = True;
                        # print("Breaking!")
                        break
            
            if started:
                # print(text)
                content_sections.append(item)
            
            # print("\n")
        
        if get_out:
            break

    return content_sections
            

# Convert the TOC hierarchies into a tree structure

TocItem: TypeAlias = list[int | str | dict]

@dataclass
class Node:
    pitem: TocItem | None = None
    pitem_idx: int = -1
    ditem: NodeItem | None = None
    parent: "Node | None" = None
    children: list["Node"] = field(default_factory=list)
    text: str = ""
    vector: list[float] | None = None
    embedding_sum: np.ndarray = None
    embedding_weight: float = 0.0
    
    
    
def make_doc_tree(pdf_file, node):
    children = []

    if node.pitem: # TODO - change to checking parent.parent
        parent_level = node.pitem[0]
    else:
        parent_level = 0
        
    for idx in range(node.pitem_idx + 1, len(pdf_file.ptoc)):
        if pdf_file.ptoc[idx][0] == parent_level: # if encountered parent's level
            break

        if pdf_file.ptoc[idx][0] == parent_level + 1: # if encountered child in ToC (not grandchildren)
            child = Node(
                pitem=pdf_file.ptoc[idx],
                pitem_idx=idx,
                parent=node,
                text=pdf_file.ptoc[idx][1]
            )

            # Get the Docling sections for the ToC item
            current_parent=child
            sections = get_sections(pdf_file, idx)
            for item in sections:
                if isinstance(item, SectionHeaderItem):
                                        
                    # if same heading as TOC title, skip it
                    if is_similar_title(item.text, child.text):
                        continue

                    # Add section heading as a grandchild
                    current_parent=Node(
                        ditem=item,
                        parent=child,
                        text=item.text + "\n\n"
                    )
                    child.children.append(current_parent)
                else:
                    text=""
                    if isinstance(item, TextItem):
                        text=item.text

                    current_parent.children.append(Node(
                        ditem=item,
                        parent=current_parent,
                        text=text
                    ))
                    
            make_doc_tree(pdf_file, child)
            node.children.append(child)

for fidx in range(file_count): 
    print(f"Making tree for File: {files_to_ingest[fidx].fname} | Title: {files_to_ingest[fidx].title}")
    files_to_ingest[fidx].tree_root=Node(text=files_to_ingest[fidx].title)
    make_doc_tree(files_to_ingest[fidx], files_to_ingest[fidx].tree_root)
    



In [ ]:
from langchain_openai import OpenAIEmbeddings
from concurrent.futures import ThreadPoolExecutor, as_completed

dimension = 0

embeddings = OpenAIEmbeddings(
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://localhost:1234/v1",
    openai_api_key=getenv("LMSTUDIO_API_KEY"),
    check_embedding_ctx_length=False
)

def gen_doc_tree_embeddings(node, level=1):
    # print(f"{"> "*level}{node.text}") 
    
    vectors=[]
    weights=[]

    for child in node.children:
        gen_doc_tree_embeddings(child, level+1)
        
        vectors.append(child.embedding_sum)
        weights.append(child.embedding_weight)

    own_vector = np.asarray(
        embeddings.embed_query(node.text),
        dtype=np.float64
    )

    norm = np.linalg.norm(own_vector)

    if norm > 0:
        own_vector = own_vector / norm

    own_weight = max(len(node.text), 1)
    
    vectors.append(own_vector * own_weight)
    weights.append(own_weight)

    node.embedding_sum = np.sum(vectors, axis=0)
    node.embedding_weight = sum(weights)

    node.vector = node.embedding_sum / node.embedding_weight

    norm = np.linalg.norm(node.vector)
    if norm > 0:
        node.vector = node.vector / norm

def process_tree(fidx):
    print(f"Generating embeddings for {files_to_ingest[fidx].fname}\n")
    gen_doc_tree_embeddings(files_to_ingest[fidx].tree_root)

trial_vector = np.asarray(
    embeddings.embed_query("Trial to retrieve dimension"),
    dtype=np.float64
)

dimension = trial_vector.shape[0]

print(f"Dimension: {dimension}\n")

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(process_tree, fidx)
        for fidx in range(file_count)
    ]

    for future in as_completed(futures):
        future.result()


print("Done!")
        

In [ ]:
from neo4j import GraphDatabase

N4J_URI = getenv("NEO4J_URI")
N4J_AUTH = (getenv("NEO4J_USERNAME"), getenv("NEO4J_PASSWORD"))
N4J_DATABASE = "neo4j"
N4J_INDEX = "docNodeVectorIndex"

driver = GraphDatabase.driver(N4J_URI, auth=N4J_AUTH)
driver.verify_connectivity()

print("Connected to neo4j")

In [ ]:
# Reinit neo4j

driver.execute_query(
    """
    MATCH (n)
    DETACH DELETE n
    """,
    database=N4J_DATABASE,
)

print("Cleared existing graph data")

driver.execute_query(
    f"""
    CREATE VECTOR INDEX {N4J_INDEX} IF NOT EXISTS
    FOR (n:DocNode)
    ON n.vector
    OPTIONS {{
        indexConfig: {{
            `vector.dimensions`: $dimension,
            `vector.similarity_function`: 'cosine'
            }}
        }}
    """,
    dimension=dimension,
    database_=N4J_DATABASE,
)

print("Created vector index")

driver.execute_query("""
    CREATE CONSTRAINT book_id IF NOT EXISTS
    FOR (b:Book)
    REQUIRE b.book_id IS UNIQUE
""", database_=N4J_DATABASE)

driver.execute_query("""
    CREATE CONSTRAINT doc_node_id IF NOT EXISTS
    FOR (n:DocNode)
    REQUIRE n.node_id IS UNIQUE
""", database_=N4J_DATABASE)

print("Created constriants")

In [ ]:


def traverse(node, book_id, node_id=None, parent_id=None):
    if not node_id:
        node_id = f"{book_id}"
        
    nodes.append({
        "node_id": node_id,
        "book_id": book_id,
        "text": node.text,
        "is_toc": node.pitem is not None,
        "vector": list(node.vector),
        "embedding_sum": node.embedding_sum.tolist(),
        "embedding_weight": node.embedding_weight
    })

    if parent_id is not None:
        relationships.append({
            "parent_id": parent_id,
            "child_id": node_id,
            "position": int(node_id.split(".")[-1]),
        })

    for i, child in enumerate(node.children):
        child_path = f"{node_id}.{i}"
        traverse(child, book_id, child_path, node_id)
    
for fidx in range(file_count): 
    nodes = []
    relationships = []
    
    print(f"Saving tree for {files_to_ingest[fidx].fname} | {files_to_ingest[fidx].title}")
    records, _, _ = driver.execute_query(
        """
        MATCH (b:Book)
        RETURN max(b.book_id) AS last_id
        """,
        database_=N4J_DATABASE,
    )
    
    last_id = records[0]["last_id"]
    next_id = 0 if last_id is None else last_id + 1
    
    print(f"Traversing book ID {next_id}...")

    traverse(files_to_ingest[fidx].tree_root, next_id)

    print(f"Saving hierarchy...")
    driver.execute_query("""
        MERGE (b:Book {book_id: $book_id})
        SET b.title = $title
            
        WITH b
        UNWIND $nodes AS item
    
        MERGE (n:DocNode {node_id: item.node_id})
        SET n.text = item.text,
            n.is_toc = item.is_toc,
            n.vector = item.vector,
            n.embedding_sum = item.embedding_sum,
            n.embedding_weight = item.embedding_weight,
            n.book_id = item.book_id
    
        WITH b
        MATCH (root:DocNode {node_id: $root_id})
        MERGE (b)-[:HAS_ROOT]->(root)
    """,
        book_id=next_id,
        title=files_to_ingest[fidx].title,
        nodes=nodes,
        root_id=nodes[0]["node_id"],
        database_=N4J_DATABASE
    )
    
    print(f"Saving relations...")

    driver.execute_query("""
        UNWIND $relationships AS rel

        MATCH (parent:DocNode {node_id: rel.parent_id})
        MATCH (child:DocNode {node_id: rel.child_id})

        MERGE (parent)-[r:HAS_CHILD]->(child)
        SET r.position = rel.position
    """,
        relationships=relationships,
        database_=N4J_DATABASE            
    )                 
    
    print("Saved\n")

    

